# Session 9: Loss Functions and Output Heads

This notebook runs the whole Session 9 assignment from top to bottom on a small decoder-only language model trained from scratch.

- **Model:** 8 layers, D=384, 6 heads, context 256, RoPE, RMSNorm, pre-norm, SwiGLU, ~20.5M parameters
- **Data:** 20M training tokens streamed from FineWeb-Edu, with our own 8,192-token BPE tokenizer
- **Hardware used:** NVIDIA RTX 5060 Laptop GPU (8 GB), Windows 11

The code lives in `.py` files in this repository. Each cell below runs one script with this notebook's Python, streams its output here, and saves a copy in `runs/notebook/`. Total run time is about 12-15 minutes, most of it Part 2 training.

The results and their interpretation are written up in [README.md](README.md).

## 0. Setup

Open this notebook from the `mini_llm` folder and select the project's `.venv` as the kernel.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / 'config.py').exists(), 'Open this notebook from the mini_llm folder (next to config.py).'
LOG_DIR = ROOT / 'runs' / 'notebook'
LOG_DIR.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHONUNBUFFERED': '1', 'TQDM_DISABLE': '1'}


def run(cmd, log_name):
    '''Run python <cmd> with this notebook kernel, stream the output, save a copy to runs/notebook/.'''
    print('$ python ' + cmd)
    print()
    args = [sys.executable] + cmd.split()
    with open(LOG_DIR / log_name, 'w', encoding='utf-8') as log, subprocess.Popen(
            args, cwd=ROOT, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace') as proc:
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
    if proc.returncode != 0:
        raise RuntimeError(f'{cmd} failed with exit code {proc.returncode}')
    print(f'[saved runs/notebook/{log_name}]')


print('Python :', sys.executable)
print('Project:', ROOT)

In [ ]:
run('check_env.py', '00_check_env.txt')

## 1. Data

Streams FineWeb-Edu, trains the 8,192-token BPE tokenizer (`<pad>` = 0, `<eos>` = 1 after every document), and writes `train.bin` and `val.bin`. If `data/prepared/` already exists this step is skipped.

In [ ]:
run('-m data.prepare_data', '01_prepare_data.txt')
meta = json.loads((ROOT / 'data' / 'prepared' / 'meta.json').read_text())
meta

## 2. Model self-test

Parameter breakdown and one forward pass on random tokens: `tokens [B,T] -> hidden [B,T,D] -> logits [B,T,V]`.

In [ ]:
run('-m model.model', '02_model_selftest.txt')

## 3. Part 1: the loss harness

`experiments/session09.py`, about 2-3 minutes:

1. Every tensor shape from tokens to the scalar loss
2. The next-token shift, printed as token strings, including a document boundary
3. Untrained loss and perplexity compared with ln(V)
4. Why the shift matters: 150 training steps with the correct shift vs. no shift
5. Padding mask: contributing-token counts and three ways to average
6. Document-boundary mask: loss before and after
7. Tied vs. untied output head: parameter counts
8. Ordinary vs. chunked cross-entropy: same loss, peak GPU memory, bf16 vs. fp32 gradients
9. Summary of the seven Part 1 numbers

Sections 5, 6 and 8 use the model trained for 150 steps in section 4.

In [ ]:
run('-m experiments.session09', '03_part1.txt')

## 4. Part 2: a second head predicting t+2

`experiments/session09_part2.py`, about 8 minutes. Two runs from identical starting weights on identical batches, 1,500 steps each:

- **baseline:** shared trunk + head 1 (t+1), loss = L1
- **mtp:** shared trunk + head 1 (t+1) + head 2 (t+2), loss = L1 + L2

Validation loss and top-1 accuracy for each head are measured every 100 steps on fixed batches, with document boundaries masked. Set `RUN_PART2 = False` to reuse the saved log and plot instead of retraining.

In [ ]:
RUN_PART2 = True   # False: reuse runs/session09_part2/log.csv and losses.png from an earlier run

if RUN_PART2:
    run('-m experiments.session09_part2', '04_part2.txt')
else:
    print('Skipped training; using the saved results in runs/session09_part2/.')

In [ ]:
import pandas as pd
from IPython.display import Image, display

log = pd.read_csv(ROOT / 'runs' / 'session09_part2' / 'log.csv')

every_500 = log[log.step % 500 == 0].pivot(index='step', columns='run', values=['val_l1', 'val_l2'])
display(every_500.round(4))

final = log[log.step == log.step.max()].set_index('run')[['val_l1', 'val_l2', 'val_acc1_pct', 'val_acc2_pct']]
final['gap_l2_minus_l1'] = final.val_l2 - final.val_l1
display(final.round(4))

png = ROOT / 'runs' / 'session09_part2' / 'losses.png'
if png.exists():
    display(Image(filename=str(png)))
else:
    print('No plot: run  uv add matplotlib  and rerun Part 2.')

## 5. Results

The numbers and what they show are written up in [README.md](README.md). Values can differ by about 0.01-0.02 between runs because some GPU kernels are not bit-for-bit deterministic.